(z3-knight)=
# ナイトツアー ― すべてのマスを1回ずつ訪れるルートを探索します

## この章で学ぶこと

チェスのナイト（騎士）が、8×8 の盤面のすべてのマスを**1回ずつ**訪れるルートを探索します。最後はスタート地点に戻る「閉じたルート」とします。

- 各マスに「何番目に訪れたか」を表す整数を割り当てます
- 「隣接する番号はナイトの移動先に存在する」を `PbEq()` で記述します
- ルールをそのまま制約にすると、Z3 が数秒でルートを見つけます

ナイトツアー問題は、[OrToolsでナイトツアー問題を解く](ortools-knight) でも扱っています。

## 考え方

各マスに `0` から `63` までの整数を割り当て、**その数が「何番目に訪れたか」**を表すようにします。

- すべてのマスの値は 0〜63 であり、すべて異なります（`Distinct`）
- 値が `c` のマスの隣接マスのどこかに、値 `c+1` のマスが存在します
- 値が `c` のマスの隣接マスのどこかに、値 `c-1` のマスが存在します
- 最後（63番目）のマスの隣接マスに、0番目のマスが存在します（閉じたルートとするため）

「隣接」は、ナイトの動き（L字型の8方向）で移動できるマスだけを指します。

この制約は、ナイトが移動できる範囲でしか満たせないため、自然にルートが決定されます。

In [1]:
from z3 import *
from itertools import product

# ナイトの8方向の動き
KNIGHT_MOVES = [(1, 2), (2, 1), (-1, 2), (2, -1), (-2, 1), (1, -2), (-1, -2), (-2, -1)]

## コード

盤面の各マスに整数変数を作成し、制約を追加します。

In [2]:
w, h = 8, 8
N = w * h

cells = {}
solver = Solver()

for i, j in product(range(h), range(w)):
    c = cells[i, j] = Int(f'c_{i}_{j}')
    solver.add(c >= 0, c <= N - 1)

# 訪れる順番はすべて異なる
solver.add(Distinct(list(cells.values())))

for i, j in product(range(h), range(w)):
    c = cells[i, j]
    neighbours = [cells[i + di, j + dj] for di, dj in KNIGHT_MOVES
                  if (i + di, j + dj) in cells]

    next_is = [(x == c + 1, 1) for x in neighbours]   # となりに c+1 があるか
    prev_is = [(x == c - 1, 1) for x in neighbours]   # となりに c-1 があるか
    start_is = [(x == 0, 1) for x in neighbours]      # となりに 0 があるか

    # 最後のマス（63）以外は、となるどこに c+1 がある
    solver.add(Implies(c < N - 1, PbEq(next_is, 1)))
    # 最初のマス（0）以外は、となるどこに c-1 がある
    solver.add(Implies(c > 0, PbEq(prev_is, 1)))
    # 最後のマス（63）は、となるどこに 0 がある（閉じたルート）
    solver.add(Implies(c == N - 1, PbEq(start_is, 1)))

%time solver.check()
m = solver.model()

for i in range(h):
    for j in range(w):
        print(f'{m[cells[i, j]].as_long():2d}', end=' ')
    print()

CPU times: total: 1.86 s
Wall time: 1.88 s
63 60 45 10 47 58 23 20 
44 11 62 59 24 21 48 57 
61  0  9 46 27 56 19 22 
12 43 36  3 54 25 28 49 
37  8  1 26 35  4 55 18 
42 13 40 53  2 31 50 29 
 7 38 15 34  5 52 17 32 
14 41  6 39 16 33 30 51 


表のように、各マスに「何番目に訪れたか」が格納されました。**0 から 63 までが1回ずつ**現れていることを確認してください。

## 見つけたルートを検証する

Z3 が出力した答えが本当にナイトツアーになっているか、自ら検証するコードも記述します。

In [3]:
def check_tour(board):
    """各マスの値を、ナイトの移動順に並べて確認する"""
    h, w = len(board), len(board[0])
    pos = {board[i][j]: (i, j) for i in range(h) for j in range(w)}

    if sorted(pos) != list(range(h * w)):
        return "番号が 0〜63 の1回ずつになっていません"

    for k in range(h * w):
        i, j = pos[k]
        i2, j2 = pos[(k + 1) % (h * w)]
        if (i2 - i, j2 - j) not in KNIGHT_MOVES:
            return f"{k} 番目から {k + 1} 番目がナイトの動きになっていません"

    return "OK: ナイトの動きで 0 → 1 → … → 63 → 0 と一周しています"


tour = [[m[cells[i, j]].as_long() for j in range(w)] for i in range(h)]
print(check_tour(tour))

OK: ナイトの動きで 0 → 1 → … → 63 → 0 と一周しています


## 制約の意味

| 制約 | 意味 |
|---|---|
| `Implies(c < N - 1, PbEq(next_is, 1))` | 最後以外のマスは、隣接するどこかに `c+1` のマスが存在します |
| `Implies(c > 0, PbEq(prev_is, 1))` | 最初以外のマスは、隣接するどこかに `c-1` のマスが存在します |
| `Implies(c == N - 1, PbEq(start_is, 1))` | 最後のマスは、隣接するどこかに `0` のマスが存在します |
| `Distinct(...)` | 0〜63 が重複しません |

「`c+1` のマスが存在する」と「`c-1` のマスが存在する」は、片方だけでも同じ意味になります（`c+1` のマスから見れば `c` は `c-1` です）。両方を記述しておくと、Z3 が探索しやすくなります。

## `PbEq()` とは

`PbEq(args, k)` は、**重み付きの合計がちょうど `k` になる**という制約です。`args` は `(条件, 重み)` のリストであり、条件が `True` なら重みが加算されます。

例：重みが 1 の条件が2つ真、重み 2 が1つ真なら合計は 4 です。

In [4]:
x, y, z = Bools('x y z')

s = Solver()
s.add(PbEq([(x, 1), (y, 2), (z, 3)], 5))
s.check()
print(s.model())

[y = True, z = True, x = False]


`x` が1点、`y` が2点、`z` が3点であり、合計がちょうど 5 になる組み合わせが選ばれました（`y` と `z` が真）。

ナイトツアーのコードでは、どの条件の重みも 1 にしています。つまり `PbEq(..., 1)` は**「条件が真になるのはちょうど1つ」**という意味になります。

`Sum(...) == 1` でも同じことを記述できますが、`PbEq()` のほうが「ちょうど1つ」という意図が明確になり、Z3 も速く解けます。

## まとめ

- 移動の順序を各マスの整数にすると、「隣接するマスどうしがナイトの移動先である」という制約でルートが決定されます
- `Distinct` で 0〜63 を1回ずつ使用することを保証します
- 「ちょうど1つ」は `PbEq([...], 1)` で記述できます
- 得られた答えは、自ら検証するコードを用意すると安心です（`check_tour()`）